# 3 · Train and predict

Train on Dynamic World, then predict land cover for the KHDTK forest from notebook 2.

The flow is: **load tiles → train → predict chips → save maps**.

In [ ]:
import warnings
from pathlib import Path

import kornia.augmentation as K
import lightning as L
import torch
from huggingface_hub import snapshot_download
from torch.utils.data import DataLoader, Dataset

import geosave as gs

L.seed_everything(0)
warnings.filterwarnings("ignore", ".*does not have many workers.*")

root = Path(
    snapshot_download(
        "fatmur/dynamicworld-300", repo_type="dataset", local_dir="data/dynamic_world"
    )
)

## Load the classes

Use the dataset's class names, colours and ignore index throughout this notebook.

In [ ]:
import yaml

classes = yaml.safe_load((root / "classes.yaml").read_text())
IGNORE_INDEX = classes["ignore_index"]
CLASS_MAP = classes["class_map"]
COLOR_MAP = classes["color_map"]
NUM_CLASSES = len(CLASS_MAP)
CLASS_MAP

## Read the data

`DynamicWorld` takes image paths and optional label paths. It groups files by place
and date. Set `chip_size` to read fixed windows instead of complete tiles.

Each item is an `(image, label)` tensor pair, or an image tensor when labels are absent.
The Dataset keeps the layout so it can merge predictions later.

In [ ]:
def preprocess(tree):
    """Mask, scale and convert an image and optional label into tensors."""
    image = tree["sentinel-2-l2a"].dataset
    image = image.gs.mask_and_scale()
    image = image.gs.to_tensor(dtype="float32")
    image = image.nan_to_num()
    if "label" not in tree.children:
        return image

    label = tree["label"].dataset["label"]
    label = label.gs.mask_and_scale().gs.to_tensor(dtype="float32")
    return image, label.nan_to_num(IGNORE_INDEX).long()


class DynamicWorld(Dataset):
    def __init__(
        self, images, labels=None, *, chip_size=None, overlap=0, resolution=10
    ):
        self.images = list(images)
        self.labels = [] if labels is None else list(labels)
        self.resolution = resolution
        self.catalog = self._create_catalog()
        samples = self.catalog.gs.sample()
        self.frames = samples.gs.frame(
            windows={"sentinel-2-l2a": {"slots": ["0D"]}},
            reference="label" if self.labels else None,
        )
        self.rows = self.frames
        if chip_size is not None:
            self.rows = self.frames.gs.chip(
                chip_size, resolution=resolution, overlap=overlap
            )

    def _create_catalog(self):
        items = []
        for path in self.images:
            items.extend(
                gs.stac.create_items(
                    {"image": path},
                    collection="sentinel-2-l2a",
                    id_format=Path(path).stem,
                )
            )
        for path in self.labels:
            items.extend(
                gs.stac.create_items(
                    {"label": path}, collection="label", id_format=Path(path).stem
                )
            )
        return gs.stac.create_catalog(items)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows.iloc[index]
        # Chips already record their pixel grid and resolution.
        resolution = self.resolution if row.gs.geobox is None else None
        tree = row.gs.to_xarray(resolution=resolution)
        return preprocess(tree)

    def merge(self, logits, *, taper=None):
        """Restore scores predicted in Dataset order to the complete frame grids."""
        if len(logits) != len(self.rows):
            raise ValueError(
                "Predict every chip before merging; do not limit evaluation batches."
            )
        outputs = {}
        for index, values in zip(self.rows.index, logits, strict=True):
            outputs[index] = values
        bands = [f"score_{channel}" for channel in range(logits.shape[1])]
        return self.frames.gs.merge(self.rows, outputs, bands=bands, taper=taper)

    def read_label(self, index, *, target):
        """Read a frame's reference label on the restored score grid."""
        row = self.frames.loc[index]
        tree = row.gs.to_xarray(collections=["label"], target=target)
        label = tree["label"].dataset["label"]
        label = label.gs.mask_and_scale().gs.to_tensor(dtype="float32")
        return label.nan_to_num(IGNORE_INDEX).long()

## Prepare the batches

Training reads complete tiles and applies matching random crops and flips to images
and labels. Validation and test read overlapping chips with the same Dataset.
All three DataLoaders return tensor pairs.

Training tiles must have matching sizes and be at least as large as `crop_size`.
The short validation run uses the first two complete tiles.

In [ ]:
class DynamicWorldDataModule(L.LightningDataModule):
    def __init__(self, root="data/dynamic_world", batch_size=8, crop_size=128):
        super().__init__()
        self.root = Path(root)
        self.batch_size = batch_size
        self.crop_size = crop_size
        self.augment = K.AugmentationSequential(
            K.RandomCrop((crop_size, crop_size)),
            K.RandomHorizontalFlip(),
            K.RandomVerticalFlip(),
            data_keys=["input", "mask"],
            keepdim=True,
        )
        self.train_dataset = None
        self.val_dataset = None
        self.test_dataset = None

    def setup(self, stage=None):
        if stage in (None, "fit") and self.train_dataset is None:
            self.train_dataset = self._create_dataset("train")
        if stage in (None, "fit", "validate") and self.val_dataset is None:
            self.val_dataset = self._create_dataset("val")
        if stage in (None, "test") and self.test_dataset is None:
            self.test_dataset = self._create_dataset("test")

    def _create_dataset(self, split):
        images = sorted((self.root / split / "images").glob("*.tif"))
        labels = sorted((self.root / split / "labels").glob("*.tif"))
        if split == "val":
            images = images[:2]
            labels = labels[:2]
        chip_size = None if split == "train" else self.crop_size
        return DynamicWorld(
            images, labels, chip_size=chip_size, overlap=self.crop_size // 2
        )

    def train_dataloader(self):
        assert self.train_dataset is not None
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True)

    def val_dataloader(self):
        assert self.val_dataset is not None
        return DataLoader(self.val_dataset, batch_size=self.batch_size)

    def test_dataloader(self):
        assert self.test_dataset is not None
        return DataLoader(self.test_dataset, batch_size=self.batch_size)

    def on_after_batch_transfer(self, batch, dataloader_idx):
        if self.trainer is None or not self.trainer.training:
            return batch
        image, label = batch
        self.augment.to(image.device)
        image, mask = self.augment(image, label.unsqueeze(1).float())
        return image, mask.squeeze(1).long()

Create the training data and inspect one image/label pair before training.

In [ ]:
data = DynamicWorldDataModule(root)
data.setup("fit")

row = data.train_dataset.frames.iloc[0]
tree = row.gs.to_xarray(resolution=10).isel(time=0)
tree.gs.plot(cols=2)

## Build the model

The model receives ordinary tensor batches. Training uses 128 × 128 crops.
Validation and test collect chip scores, then ask the Dataset to merge them at the
end of each epoch. Metrics count each original valid pixel once.

Keep evaluation loaders in Dataset order and run every chip. Sanity validation is
disabled below because it only runs a few batches and cannot reconstruct complete maps.

In [ ]:
from torch import nn
from geosave.model import build_model
from torchmetrics import MetricCollection
from torchmetrics.classification import MulticlassAccuracy, MulticlassJaccardIndex


class Segmenter(L.LightningModule):
    def __init__(
        self, num_classes, ignore_index, in_channels=4, input_size=128, width=16
    ):
        super().__init__()
        self.model = build_model(
            {
                "encoder": {
                    "name": "small_cnn",
                    "init_args": {
                        "in_channels": in_channels,
                        "input_size": input_size,
                        "width": width,
                    },
                },
                "decoder": {"name": "unet"},
                "head": {
                    "name": "segmentation",
                    "init_args": {"num_classes": num_classes},
                },
            }
        )
        self.input_size = input_size
        self.ignore_index = ignore_index
        self.val_metrics = MetricCollection(
            {
                "miou": MulticlassJaccardIndex(
                    num_classes=num_classes, ignore_index=ignore_index
                ),
                "accuracy": MulticlassAccuracy(
                    num_classes=num_classes, ignore_index=ignore_index, average="micro"
                ),
            },
            prefix="val_",
        )
        self.test_metrics = self.val_metrics.clone(prefix="test_")
        self.criterion = nn.CrossEntropyLoss(ignore_index=ignore_index)
        self.val_outputs = []
        self.test_outputs = []

    def forward(self, image):
        return self.model(image=image)

    def compute_loss(self, logits, label):
        if label.ne(self.ignore_index).any():
            return self.criterion(logits, label)
        return logits.sum() * 0

    def training_step(self, batch, batch_index):
        image, label = batch  # (B, C, H, W), (B, H, W)
        logits = self(image)  # (B, classes, H, W)
        loss = self.compute_loss(logits, label)
        self.log("train_loss", loss)
        return loss

    def validation_step(self, batch, batch_index):
        image, _ = batch
        self.val_outputs.append(self(image).cpu())

    def test_step(self, batch, batch_index):
        image, _ = batch
        self.test_outputs.append(self(image).cpu())

    def evaluate(self, dataset, batches, metrics, prefix):
        logits = torch.cat(batches).numpy()
        for index, raster in dataset.merge(logits).items():
            scores = raster.gs.to_tensor(dtype="float32").to(self.device)
            label = dataset.read_label(index, target=raster).to(self.device)
            label = label.masked_fill(~scores.isfinite().all(dim=0), self.ignore_index)
            scores = scores.nan_to_num().unsqueeze(0)
            label = label.unsqueeze(0)
            self.log(f"{prefix}_loss", self.compute_loss(scores, label), batch_size=1)
            metrics.update(scores, label)
        self.log_dict(metrics, on_step=False, on_epoch=True)
        batches.clear()

    def on_validation_epoch_end(self):
        dataset = self.trainer.datamodule.val_dataset
        self.evaluate(dataset, self.val_outputs, self.val_metrics, "val")

    def on_test_epoch_end(self):
        dataset = self.trainer.datamodule.test_dataset
        self.evaluate(dataset, self.test_outputs, self.test_metrics, "test")

    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters(), lr=1e-2)


module = Segmenter(
    num_classes=NUM_CLASSES,
    ignore_index=IGNORE_INDEX,
    in_channels=4,
    input_size=128,
    width=16,
)
module

## Train and evaluate

Run six short epochs. Validation covers every chip from two complete tiles.
Logs and checkpoints go to `lightning_logs/03_train/`; rerunning resumes the last checkpoint.

In [ ]:
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import TensorBoardLogger

logger = TensorBoardLogger("lightning_logs", name="03_train", version=0)
checkpoint = ModelCheckpoint(save_last=True)

trainer = L.Trainer(
    max_epochs=6,
    limit_train_batches=6,
    num_sanity_val_steps=0,
    log_every_n_steps=1,
    logger=logger,
    callbacks=[checkpoint],
)
trainer.fit(module, datamodule=data, ckpt_path="last")

Evaluate on the held-out test split.

In [ ]:
trainer.test(module, datamodule=data)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir lightning_logs

## Predict over the forest

Use the same Dataset with image paths and no labels. The model runs on ordinary
image batches without a Trainer; `merge()` restores their scores to the original dated maps.

In [ ]:
region = gs.open_vector("khdtk.geojson")
images = sorted(Path("data/khdtk/sentinel-2-l2a").glob("*.tif"))
forest = DynamicWorld(
    images, chip_size=module.input_size, overlap=module.input_size // 2
)
loader = DataLoader(forest, batch_size=data.batch_size)
module.eval()
batches = []
with torch.inference_mode():
    for image in loader:
        batches.append(module(image.to(module.device)).cpu())
logits = torch.cat(batches).numpy()
merged = forest.merge(logits)

## Choose the classes

`postprocess()` picks the winning class ID from each merged score map.
It restores dates, class colours and nodata for plotting and saving.

In [ ]:
def postprocess(merged, frames):
    """Convert merged scores into dated land-cover maps."""
    # Output channels follow ascending class IDs from classes.yaml.
    class_ids = sorted(CLASS_MAP)

    maps = []
    for index, raster in merged.items():
        # Select class IDs rather than channel positions; uncovered pixels stay nodata.
        scores = raster.to_dataarray("class_id").assign_coords(class_id=class_ids)
        labels = scores.idxmax("class_id", fill_value=IGNORE_INDEX).astype("uint8")
        # Restore the date of the frame these chips came from.
        time = frames.loc[index, "start_datetime"]
        maps.append(labels.assign_coords(time=time))

    # Assemble dated maps and attach the metadata needed for plotting and export.
    cover = gs.transform.temporal.concat(maps).rename("label")
    cover = cover.gs.write_nodata(IGNORE_INDEX)
    return cover.gs.write_legend(CLASS_MAP, COLOR_MAP)


forest_cover = postprocess(merged, forest.frames)
forest_cover.gs.plot()

## Save the maps

Write one TIFF per date and a catalog pointing to those files. Keep the catalog and raster folder together when moving the results.

In [ ]:
folder = Path("data/khdtk_pred")

cover_scenes = forest_cover.gs.to_cog(
    folder / "land_cover/{time:%Y%m%d}.tif", overwrite=True
)
cover_collection = gs.stac.create_collection(
    "land_cover",
    description="Land-cover predictions for individual acquisition days from the tutorial model",
)
cover_items = gs.stac.create_items(
    cover_scenes, collection=cover_collection, id_format="{start:%Y%m%d}"
)
cover_catalog = gs.stac.create_catalog(cover_items, collections=[cover_collection])
cover_catalog.gs.to_geoparquet(folder / "catalog.parquet", overwrite=True)
cover_catalog

In [ ]:
forest_cover.gs.explore(overlay={"KHDTK": region}, basemap="Imagery")

## Try it yourself

These exercises are optional.

### 1 · Which date has the most trees?

Count how many pixels the model called trees in each map. Which date has the largest count?

<details><summary>Hint</summary>

Trees have class ID `1`. Compare `forest_cover` with `1`, then add up the matching pixels for each date.

```python
(forest_cover == 1).sum(["y", "x"]).to_series()
```
</details>

In [ ]:
# Your code here

### 2 · What if we only count inside the forest?

The maps also include land outside the forest outline. Count the tree pixels again, this time only inside the outline. How do the counts compare with the first exercise?

<details><summary>Hint</summary>

Make a mask from the forest outline using the prediction map's grid. Keep the pixels inside it, then count the trees as before.

```python
inside = region.gs.rasterize(forest_cover)
(forest_cover.where(inside) == 1).sum(["y", "x"]).to_series()
```
</details>

In [ ]:
# Your code here

### 3 · Would a bigger model help?

Train a slightly bigger version of the model using the same images. Does its validation accuracy improve?

<details><summary>Hint</summary>

Create `Segmenter(num_classes=NUM_CLASSES, ignore_index=IGNORE_INDEX, width=32)` instead of using the default width of `16`. Keep the other settings the same so you can compare the two runs.

Use a new name such as `03_wider` in `TensorBoardLogger`, then create a new `Segmenter` and Trainer. This starts a fresh run instead of resuming the earlier checkpoint.
</details>

In [ ]:
# Your code here

### 4 · Can you make the joins less visible?

Try a tapered merge, which gives more weight to the middle of each chip.

<details><summary>Hint</summary>

The forest chips already overlap. Reuse their predictions with a Hann window:

```python
merged = forest.merge(logits, taper="hann")
forest_cover = postprocess(merged, forest.frames)
forest_cover.gs.plot()
```

Compare the same date before and after. You do not need to run the model again.
</details>

In [ ]:
# Your code here